# Tutorial 03: Core Phenotyping Workflow

**About this tutorial**
- **Time:** ~30–45 minutes
- **Prerequisites:** [Getting started](../../docs/src/content/docs/getting-started.md); optional [Tutorial 02](02_inspect_your_images.ipynb)
- **Key ideas:** `find_files`, `process_images`, properties, distances, report, saving results

**Before you start:** Make sure you are running this from the repository (e.g., under `Notebooks/tutorials/`). This tutorial is the main programmatic entry point for understanding the physical and mathematical properties of your cells.

**In plain terms:** The AI acts like a highly trained observer, summarizing the appearance of each image into a mathematical profile called an **embedding**. **t-SNE** plots help you see whether different treatments (like drugs) separate into distinct groups based on these profiles. By looking at the **distance to a reference** (e.g., untreated control cells), you can quantify how drastically a treatment altered the cells. We then combine these AI summaries with **hand-crafted properties** (like size or brightness) to give you a complete biological picture.

This notebook covers:
1. Core workflow (Find Images, Process, Measure Traits, Compare Distances)
2. Automated **HTML report** generation
3. Basic **Explainability** (Connecting math back to biology)
4. Saving your results and configurations

See [Getting Started](../../docs/src/content/docs/getting-started.md) and [Common Workflows](../../docs/src/content/docs/workflows.md) for more details.

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1.1. Optional: Setup Toy Dataset (E. coli)

If you don't have your own data yet, you can download a sample dataset of E. coli images treated with different drugs.

In [ ]:
import requests
import zipfile
from pathlib import Path

def download_toy_dataset(output_dir="./toy_dataset"):
    output_path = Path(output_dir)
    if (output_path / "Ecoli_images").exists():
        print(f"Dataset already exists at {output_path}")
        return output_path

    url = "https://upvehueus-my.sharepoint.com/:u:/g/personal/aitor_gonzalezm_ehu_eus/IQCBYEL7E_-NTZZeIsBhwwG8AbI9BucCBAgaPacYqXjDsA4?download=1"
    output_path.mkdir(exist_ok=True, parents=True)
    zip_path = output_path / "ecoli_dataset.zip"
    
    print("Downloading E. coli dataset...")
    r = requests.get(url, allow_redirects=True)
    with open(zip_path, "wb") as f:
        f.write(r.content)
    
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_path)
    zip_path.unlink()
    
    # Find the actual images dir (might be nested)
    extracted_images = list(output_path.glob("**/Ecoli_images"))
    return extracted_images[0].parent if extracted_images else output_path

# Uncomment to download
# dataset_root = download_toy_dataset()
# image_dir = dataset_root / "Ecoli_images"
# mask_dir = dataset_root / "Ecoli_masks"

## 1. Setup and Reproducibility

Use `PhenoMe(seed=42)` for reproducibility. For full experiment tracking, use `export_experiment_config()` after `save_results()`.

In [ ]:
from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

## 2. Path-Based Metadata Extraction

You can automatically extract useful information just from how your folders are named. For example, if your file paths look like `data/Treated/24h/cell_001.tif`, you can use the template `.../(condition)/(timepoint)/(filename).*`. The tool will automatically learn that the condition is 'Treated' and the time is '24h'.

In [ ]:
# Template: .../(condition)/(timepoint)/(filename).*
metadata_fn = get_metadata_from_path(".../(condition)/(timepoint)/(filename).*")
file_df = pheno.find_files("path/to/images", metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")
print(f"Metadata keys: {file_df.columns.tolist()}")

## 3. Load Model and Process Images

Use `channel_mode='split'` for fluorescence images (each channel processed independently). Use `'combined'` for brightfield/RGB. See [Channel Modes](../../docs/src/content/docs/concepts.md#channel-modes).

In [ ]:
# Optional: inspect before processing
# inspect_df = pheno.inspect_data()

wrapper = load_dinov2_model(model_name="dinov2_vitg14_reg", device=device)

pheno.process_images(
    wrapper, 
    channel_mode="split",
    checkpoint_path="my_checkpoint.h5", # Recommended for large datasets. Saves embeddings to disk.
)

### 3.1. Batch Correction (optional)

If your data spans multiple plates or batches, you should correct for technical variation (plate effects) before computing distances or visualizing. This ensures that the differences you see are biological, not technical.

In [ ]:
# Correct for plate-to-plate variation using 'Control' wells as baseline
pheno.correct_batches(
    batch_metadata_key="Plate",
    control_filters={"condition": "Control"},
    method="sphering",  # Recommended for embeddings
)

## 4. Distance Analysis and Visualization (Optional)

Compare different groups of cells against a baseline, such as your `Control` group. 

**How to read the plots:** Points that cluster together in the space share a similar biological state. We measure the actual distance between these groups and the reference "centroid" (which is simply the mathematical average or center point of the Control group). The distance output tells you exactly how far a certain treatment pushed the cells from their normal state.

In [ ]:
# Compute distances to the reference group; violin plot is shown when `group_by` is set (default: plot=True)
dist_results = pheno.compute_reference_distances(
    reference_filters={"condition": "Control"},
    source="embeddings",
    mode="centroid",
    group_by="condition",
)

In [ ]:
# Visualize
pheno.plot_tsne(color_by="condition")

## 5. Measuring Physical Traits (Size, Shape, Brightness)

**Why extract physical properties?** While the AI handles subtle patterns effortlessly, classical features (like area, eccentricity, and brightness) are much easier to interpret biologically. By providing corresponding **masks** (outlines of your cells), we measure exactly what the cells look like physically.

Use `property_preset='basic'` for essential checks, or `'standard'` for exhaustive measurements. See [Property interpretation](../../docs/src/content/docs/concepts/property-interpretation.md) for what each built-in property means; for preset bundles use [Select properties](../../docs/src/content/docs/guides/select-properties/index.md).

In [ ]:
# Requires masks (mask_dir in find_files) for shape and masked-intensity features
df_props = pheno.compute_properties(property_preset="basic")
print(f"Computed {len(df_props.columns)} properties for {len(df_props)} samples")

## 5.1. Summarizing Properties

While distance analysis (Step 4) tells you *how much* a group has changed in the AI's abstract space, comparing physical traits tells you *exactly what* changed in biological terms.

In [ ]:
# Aggregate statistics for each experimental group
df_grouped = pheno.property_stats_by_group(
    group_by=["condition"],
    print_properties=["area", "eccentricity", "solidity"], # Show a few for brevity
)

## 6. Explainability: Connecting Math to Biology

The AI looks at abstract patterns to separate your cells into groups. But what do those patterns mean? This explainability step connects those hidden mathematical patterns back to real biology. By checking the correlations, you can find out if the AI is separating cells specifically based on their area, brightness, or texture. It essentially answers: *"What biological feature does this axis on my plot actually represent?"*

See [Embeddings interpretability](../../docs/src/content/docs/concepts/embeddings-interpretability.mdx) for correlation parameters and [Property interpretation](../../docs/src/content/docs/concepts/property-interpretation.md) for built-in property definitions.

In [ ]:
# Relate t-SNE components to measured properties
corr_results = pheno.compute_component_correlation(
    method="tsne",
    n_components=3,
    top_k=10
)

## 7. Save Results and Generate Report

You’re done with the main analysis.  
Now you can:

1. **Save results** (embeddings + measured properties)  
2. **Generate one interactive HTML report** to explore and share findings

### What the HTML report includes

- **Interactive plots** (zoom, pan, hover)
- **Group distance summary** (for example vs. `Control`)
- **Explainability highlights** (how embeddings relate to physical traits)
- **Outlier detection** (unusual cells)
- **Image galleries** by condition/cluster
- **No server required**: open the `.html` file directly in any browser

See the **[Report API docs](../../docs/src/content/docs/advanced/api/report.md)** for full options.

> **Note on clustering:** If `include_clustering=True` and no cluster labels exist, clustering is run automatically. For deeper clustering workflows, see [04_exploratory_analysis_and_explainability.ipynb](04_exploratory_analysis_and_explainability.ipynb).

In [ ]:
# 1. Save all embeddings and properties to an HDF5-based results structure
pheno.save_results("results.h5")

# Example: To load results later in a new session:
# pheno_new = PhenoMe(device=device, seed=42)
# pheno_new.find_files("path/to/images") # Re-map paths to the current machine
# pheno_new.load_results("results.h5")

In [ ]:
# 2. Generate an HTML report
from phenome.report import ReportConfig

# You can have finely-tuned control via the ReportConfig object 
# OR just pass kwargs directly to `generate_report`.
report_config = ReportConfig(
    include_clustering=True,
    n_clusters=5,
    include_image_gallery=True,
    include_outliers=True,
    outlier_threshold=3.0,
    top_k_features=10, 
)

print("Building standalone interactive report...")
report_path = pheno.generate_report(
    output_path="results/report.html",
    config=report_config,
    reference_filters={"condition": "Control"},
    color_by="condition",
)
print(f"Done! Open '{report_path}' in your web browser.")

## Next Steps

- [04_exploratory_analysis_and_explainability.ipynb](04_exploratory_analysis_and_explainability.ipynb): Advanced clustering, prototypes, and deep explainability
- [05_extending_phenome_plugins.ipynb](05_extending_phenome_plugins.ipynb): Writing custom plugins and model wrappers